In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [6]:
# Risk Parity, Normal Assumption
def portfolio_risk_contribution(w: np.ndarray, cov: np.ndarray) -> np.ndarray:
    # Marginal contribution to risk
    mrc = cov @ w

    # Total portfolio variance
    port_var = w.T @ cov @ w

    # Risk contribution of each asset
    rc = w * mrc
    return rc, port_var

def risk_parity_objective(w: np.ndarray, cov: np.ndarray) -> float:
    rc, port_var = portfolio_risk_contribution(w, cov)

    # Equal target contribution
    target_rc = port_var / len(w)

    # Minimize squared distance from equal risk contributions
    return np.sum((rc - target_rc) ** 2)

# Load covariance matrix
cov_input = pd.read_csv("/Users/fuyuxuan/Downloads/test5_2.csv")

# Convert to numpy array
cov_matrix = cov_input.values

# Number of assets
n = cov_matrix.shape[0]

# Initial guess: equal weights
w0 = np.ones(n) / n

# Constraint: weights sum to 1
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

# Long-only bounds
bounds = [(0, 1) for _ in range(n)]

# Optimize
result = minimize(
    risk_parity_objective,
    w0,
    args=(cov_matrix,),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-15, "maxiter": 1000}
)

# Get optimized weights
weights = result.x

output = pd.DataFrame(weights, columns=["W"])
print(output)

          W
0  0.035463
1  0.025806
2  0.056468
3  0.265924
4  0.616339
